**Usecase3: File notification (every file is created in the cloud storage will trigger the LF Ingestion job and any newly added files will be autoloaded into bronze layer incrementally)**

![](realtimestream.png)

Steps to configure autoloader with file trigger: 
Storage Account Contributor: Assigned at the Storage Account level. This allows Databricks to configure the queue and event subscriptions.

Storage Queue Data Contributor: Assigned at the Storage Account level. This allows Databricks to read and delete messages from the queue it creates.

EventGrid EventSubscription Contributor: Assigned at the Resource Group level (the resource group containing your storage account). This allows Databricks to create the Event Grid subscription.

In [0]:
# 1. Define your Azure Data Lake Storage Gen2 (ADLS Gen2) paths
source_path = "abfss://containerwd362@wd36adlsstorageacct2.dfs.core.windows.net/landing-zone/" 
target_table = "wd36_workspace1.default.cust_stream1"
checkpoint_path = "abfss://containerwd362@wd36adlsstorageacct2.dfs.core.windows.net/_checkpoints/my_pipeline/"
schema_path = "abfss://containerwd362@wd36adlsstorageacct2.dfs.core.windows.net/_schemas/my_pipeline/"

# 2. Define the Auto Loader configuration dictionary for Azure
cloudFilesConf = {
    "cloudFiles.format": "csv",
    "cloudFiles.schemaLocation": schema_path,
    "cloudFiles.useNotifications": "true",
    
    # Azure-specific configuration required to create Event Grid and Queues:
    "cloudFiles.subscriptionId": "e68a829e-83e8-42bc-8494-b4ceb80be6c7",
    "cloudFiles.resourceGroup": "wd36-rg1",
    "cloudFiles.tenantId": "089cf941-fd24-44c5-8a92-985784dc42d6",
    "cloudFiles.clientId": "530cd74e-90c7-4365-8fed-2d91d79fb006",
    "cloudFiles.clientSecret": ".rX8Q~N1tX8g.O2O~0Eov79fDAYuKYHBkKR"}

# 3. Read Stream
df = (spark.readStream
      .format("cloudFiles")
      .options(**cloudFilesConf)
      .load(source_path))

# 4. Write Stream
query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", checkpoint_path)
         .trigger(processingTime='10 seconds')
         .toTable(target_table))
#         
